# Support Vector Machine

In [3]:
import pandas as pd # type: ignore
Data_final = pd.read_csv('/Users/instructorzamora/Documents/3_Maestria_Estadistica_UNINORTE/3_Tercer_Semestre/Machine_Learning/Deteccion_Fraude/Data_final.csv')
Data_final

,D12,D14,D11,D8,TransAmt,D3,D7,dist1,dist2,V209,...,V285,id_01,D13,isFraud,card4_discover,card4_mastercard,card4_visa,card6_credit,card6_debit,card6_debit or credit
0,0.0,0.0,13.0,37.875,68.500000,13.0,0.0,19.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,1,0,0,1,0,0
1,0.0,0.0,43.0,37.875,29.000000,8.0,0.0,8.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,1,0,1,0,0
2,0.0,0.0,315.0,37.875,59.000000,8.0,0.0,287.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,0,1,0,1,0
3,0.0,0.0,43.0,37.875,50.000000,0.0,0.0,8.0,37.0,0.0,...,10.0,-5.0,0.0,0.0,0,1,0,0,1,0
4,0.0,0.0,43.0,37.875,50.000000,8.0,0.0,8.0,37.0,0.0,...,0.0,0.0,0.0,0.0,0,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,0.0,0.0,56.0,37.875,49.000000,30.0,0.0,48.0,37.0,0.0,...,1.0,-5.0,0.0,0.0,0,0,1,0,1,0
590536,0.0,0.0,0.0,37.875,39.500000,8.0,0.0,8.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,1,0,0,1,0
590537,0.0,0.0,0.0,37.875,30.950001,8.0,0.0,8.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,1,0,0,1,0
590538,0.0,0.0,22.0,37.875,117.000000,0.0,0.0,3.0,37.0,0.0,...,5.0,-5.0,0.0,0.0,0,1,0,0,1,0


El dataset final es un conjunto de datos extenso de detección de fraude con 590,540 registros y 22 columnas, diseñado para un modelo de machine learning que busca identificar transacciones fraudulentas. Contiene variables numéricas como 'TransAmt' (monto de transacción), 'dist1', 'dist2', y códigos como D12, D14, D11, junto con variables categóricas binarias que representan características de tarjetas de pago (como tipos de tarjetas Discover, Mastercard, Visa, y tipos de tarjetas de crédito/débito). La variable objetivo 'isFraud' es binaria (0 o 1), indicando si una transacción es fraudulenta, mientras que la mayoría de las otras variables son numéricas con muchos valores cercanos a cero, sugiriendo un preprocesamiento de datos previo. Este dataset parece estar preparado para entrenar un modelo de clasificación que pueda predecir la probabilidad de fraude en transacciones financieras.

## Support Vector Machine

In [4]:
# ------------------------
# Paso 1: Importar paquetes necesarios
# ------------------------
import numpy as np
import pandas as pd
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline  # SMOTE + scaling
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from joblib import dump
from time import time

# ------------------------
# Paso 2: Cargar y preparar los datos
# ------------------------
y = Data_final['isFraud']
x = Data_final.drop(columns=['isFraud']).select_dtypes(include='number')  # Solo numéricas
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.20,random_state=80,stratify=y)

# ------------------------
# Paso 3: Pipeline con SMOTE, escalado y SGDClassifier
# ------------------------
pipe_sgd_svm = Pipeline([
    ('smote', SMOTE(random_state=42, n_jobs=-1)),
    ('scaler', StandardScaler()),
    ('sgd', SGDClassifier(
        loss='hinge',           # comportamiento de SVM
        penalty='l2',           # regularización
        alpha=1e-4,             # similar a C en SVM (regularización inversa)
        max_iter=1000,          # número de épocas
        tol=1e-3,               # tolerancia para detener
        random_state=42,
        n_jobs=-1
    ))
])

# ------------------------
# Paso 4: Entrenar el modelo
# ------------------------
start_time = time()
pipe_sgd_svm.fit(x_train, y_train)
training_time_sgd = time() - start_time

# ------------------------
# Paso 5: Guardar el modelo
# ------------------------
dump(pipe_sgd_svm, 'sgd_svm_smote_optimized.joblib')

# ------------------------
# Paso 6: Predicciones
# ------------------------
y_pred_sgd = pipe_sgd_svm.predict(x_test)
y_score_sgd = pipe_sgd_svm.decision_function(x_test)

# ------------------------
# Paso 7: Métricas
# ------------------------
precision_sgd = precision_score(y_test, y_pred_sgd, average='weighted')
recall_sgd = recall_score(y_test, y_pred_sgd, average='weighted')
accuracy_sgd = accuracy_score(y_test, y_pred_sgd)
f1_sgd = f1_score(y_test, y_pred_sgd, average='weighted')
auc_sgd = roc_auc_score(y_test, y_score_sgd)

# ------------------------
# Paso 8: Resultados
# ------------------------
resultados_sgd = pd.DataFrame({
    'Precision': [f"{precision_sgd:.2f}"],
    'Recall': [f"{recall_sgd:.2f}"],
    'Accuracy': [f"{accuracy_sgd:.2f}"],
    'F1-Score': [f"{f1_sgd:.2f}"],
    'AUC': [f"{auc_sgd:.2f}"],
    'CPU time (s)': [round(training_time_sgd, 2)]
})

print("✅ Métricas SGDClassifier (SVM rápido) + SMOTE optimizado:")
display(resultados_sgd)


/Users/instructorzamora/miniconda3/envs/ml_venv/lib/python3.9/site-packages/imblearn/over_sampling/_smote/base.py:370: FutureWarning: The parameter `n_jobs` has been deprecated in 0.10 and will be removed in 0.12. You can pass an nearest neighbors estimator where `n_jobs` is already set instead.
  warnings.warn(


✅ Métricas SGDClassifier (SVM rápido) + SMOTE optimizado:


,Precision,Recall,Accuracy,F1-Score,AUC,CPU time (s)
0,0.94,0.93,0.93,0.93,0.70,2.77


El modelo de Máquinas de Vectores de Soporte (SVM) muestra un rendimiento sólido en la detección de fraudes. Con una precisión del 94%, el modelo es muy preciso en la identificación de transacciones fraudulentas. El recall del 93% indica que captura la gran mayoría de las transacciones fraudulentas reales. La accuracy del 93% confirma su buena efectividad general en la clasificación de transacciones. El F1-Score de 0.93 representa un equilibrio muy bueno entre precisión y recall. El AUC de 0.70, aunque aceptable, es el más bajo entre los modelos evaluados, sugiriendo algunas limitaciones en la separación de clases. El tiempo de CPU de apenas 4 segundos es notablemente bajo, lo que indica una gran eficiencia computacional en comparación con otros modelos, siendo potencialmente útil para implementaciones que requieren rapidez de procesamiento.